# CIFAR-10 CNN: Tucker-2 rank sweep

`01_tucker2_conv.ipynb` で1つのConv2dをTucker-2へ置き換えられた前提で、
`rank_out` / `rank_in` を複数試し、サイズと精度のtrade-offを比較する。

このNotebookでは新しい分解アルゴリズムを作らない。
01で完成した1層置換処理を再利用する。

ゴールは「どのrankが良いか」を単一指標ではなく、
**パラメータ削減とvalidation accuracyの両方から選べること**。


## 1. baseline・DataLoader・評価関数を準備する

既存の `CIFAR10CNN`、baseline checkpoint、`evaluate`、パラメータ数計測など
`src` にある共通処理を優先して使う。

CIFAR-10 DataLoaderはSVD実験で使った条件と揃える。
新しい前処理を勝手に追加して比較条件を変えない。


In [1]:
from pathlib import Path
import copy

import torch
from torch import nn

from __future__ import annotations

from copy import deepcopy
import copy
from pathlib import Path
import json
import platform
import sys

print("import: numpy / pandas / matplotlib", flush=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 評価
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# PyTorch
print("import: torch", flush=True)
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.preprocessing import MinMaxScaler

# Jupyter の cwd が notebooks/ でも src を見つける
for _candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _src = _candidate / "src"
    if (_src / "nn_compression").is_dir():
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break
# CursorでTucker処理をsrcへ共通化した後、実際のexport先からimportする。
print("import: nn_compression", flush=True)
from nn_compression.utils import (
    find_project_root,
    get_experiment_dirs,
    get_named_module,
    set_seed,
)
from nn_compression.training import (
    evaluate,
    fit_with_early_stopping,
    non_shuffling_loader,
    train_one_epoch,
)
from nn_compression.selection import (
    extract_pareto_frontier,
    find_knee_point,
    get_first_point,
    line_equation,
)
from nn_compression.metrics import (
    accuracy_drop,
    agreement,
    benchmark_inference,
    collect_compression_metrics,
    take_inference_batch,
    count_parameters,
    estimate_cnn_macs,
    estimate_conv2d_macs,
    logits_rmse,
    parameters_reduction,
)
from nn_compression.compression import (
    factorize_conv2d_layer,
    factorize_named_layers,
    retained_energy,
    sweep_conv_svd_ranks,
)
from nn_compression.models import CIFAR10CNN
from nn_compression.datasets import shuffled_index_splits
from nn_compression.compression import (
    hosvd,
    reconstruct_tucker,
    tucker_parameter_count,
)
from nn_compression.metrics import (
    relative_frobenius_error,
)
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "nn-compression-svd-dmrg" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models/10_svd/40_cifar10_cnn/02_svd_global_compression_using_src_corrected"
    / "cifar10_cnn_baseline.pt"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


import: numpy / pandas / matplotlib
import: torch
import: nn_compression


device(type='cuda')

In [2]:
# 再現性のため乱数seedを固定する
SEED = 0
set_seed(SEED)

# CUDA → MPS → CPU の順で利用可能なデバイスを選ぶ
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"device: {device}")
print(f"PyTorch: {torch.__version__}")

device: cuda
PyTorch: 2.11.0+cu128


In [3]:
# method-first の新しい構成に合わせて保存先を識別する。
METHOD_NAME = "20_tucker"
CASE_NAME = "10_cifar10_cnn"
# 原本の results/models を上書きしない
EXPERIMENT_NAME = "02_rank_sweep.ipynb"

project_root = find_project_root(Path.cwd())
data_dir, models_dir, results_dir = get_experiment_dirs(
    project_root,
    METHOD_NAME,
    CASE_NAME,
    EXPERIMENT_NAME,
)

checkpoint_dir = models_dir
csv_dir = results_dir
figure_dir = results_dir
metadata_dir = results_dir

print("project_root:", project_root)
print("data_dir:", data_dir)
print("models_dir:", models_dir)
print("results_dir:", results_dir)


project_root: D:\dev\nn-compression-svd-dmrg
data_dir: D:\dev\nn-compression-svd-dmrg\data
models_dir: D:\dev\nn-compression-svd-dmrg\models\20_tucker\10_cifar10_cnn\02_rank_sweep.ipynb
results_dir: D:\dev\nn-compression-svd-dmrg\results\20_tucker\10_cifar10_cnn\02_rank_sweep.ipynb


In [4]:
import torchvision.transforms as transforms

# 学習データ専用の前処理（Data Augmentation を含む）
#transforms.Compose([...]) は、リスト中の変換を上から順番に連結して実行する仕組みです。
train_transform = transforms.Compose(
    [
        # 元画像（32 x 32）の周囲を4 pixelぶんゼロ埋めして40 x 40にして
        # 領域内からランダムな位置で 32 x 32 を切り出す。
        # 物体の位置が少しずれても認識できるように学習させる。
        transforms.RandomCrop(
            size=32,
            padding=4,
        ),

        # 確率 p=0.5（デフォルト）で画像を左右反転する。
        # CIFAR-10では、左右が反転しても通常はクラスが変わらないため有効。
        # 例: 左向きの車も右向きの車も automobile として学習する。
        transforms.RandomHorizontalFlip(),

        # PIL Image / NumPy配列をPyTorch Tensorに変換する。
        # 形状: (H, W, C) = (32, 32, 3) -> (C, H, W) = (3, 32, 32)
        # 型・値域: uint8 の [0, 255] -> float32 の [0.0, 1.0]
        transforms.ToTensor(),

        # RGB各チャネルを同じ平均・標準偏差で正規化する。
        # 各画素値 x (ToTensor後は [0, 1]) に対し、
        # x_normalized = (x - 0.5) / 0.5 を適用する。
        # その結果、値域は概ね [0, 1] -> [-1, 1] となる。
        transforms.Normalize(
            mean=(0.5, 0.5, 0.5),  # R, G, B の平均との差し引き用
            std=(0.5, 0.5, 0.5),   # R, G, B のスケール調整用
        ),
    ]
)


# 検証・テスト専用の前処理
evaluation_transform = transforms.Compose(
    [
        # 評価時もCNNに渡せるTensor形式へ変換する。
        # (32, 32, 3) -> (3, 32, 32)、[0, 255] -> [0.0, 1.0]
        transforms.ToTensor(),

        # 学習時と「まったく同じ」正規化を行う。
        # 学習と評価でスケールが違うと、入力分布が変わって正しい評価にならない。
        transforms.Normalize(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
        ),
    ]
)

In [5]:
# ============================================================
# CIFAR-10を最初の1回だけダウンロードする
# ============================================================

full_train_augmented = datasets.CIFAR10(
    root=data_dir,
    train=True,
    # 今回はミラーサイトから手動でDLしたためFalse
    download=False,
    transform=train_transform,
)

# すでにダウンロード済みなので download=False
full_train_evaluation = datasets.CIFAR10(
    root=data_dir,
    train=True,
    download=False,
    transform=evaluation_transform,
)

# testデータも同じCIFAR-10アーカイブ内に含まれている
test_dataset = datasets.CIFAR10(
    root=data_dir,
    train=False,
    download=False,
    transform=evaluation_transform,
)

TRAIN_SIZE = 40_000
VALIDATION_SIZE = 5_000

# ------------------------------------------------------------
# train / validation のindexを固定
# SEED が同じなら randperm 結果も同一になり、割当を再現できる。
# 【.py】datasets/splits.py。分割長だけ実験条件
# ------------------------------------------------------------
train_indices, validation_indices_early_stop, validation_indices_rank = (
    shuffled_index_splits(
        len(full_train_augmented),
        (TRAIN_SIZE, VALIDATION_SIZE, VALIDATION_SIZE),
        seed=SEED,
    )
)

# ------------------------------------------------------------
# 同じ元データだがtransformを変える
# ------------------------------------------------------------

# train: augmentationあり
train_dataset = Subset(
    dataset=full_train_augmented,
    indices=train_indices,
)

# validation: augmentationなし
validation_dataset_early_stop = Subset(
    dataset=full_train_evaluation,
    indices=validation_indices_early_stop,
)

validation_dataset_rank = Subset(
    dataset=full_train_evaluation,
    indices=validation_indices_rank,
)

print(f"train:      {len(train_dataset):,}")
print(f"validation: {len(validation_dataset_early_stop):,}")
print(f"validation: {len(validation_dataset_rank):,}")
print(f"test:       {len(test_dataset):,}")


train:      40,000
validation: 5,000
validation: 5,000
test:       10,000


## 2. 01で完成したTucker-2 Conv置換処理を使えるようにする

01の実装がまだNotebook内だけなら一時的にコピーしてよい。

ただし、このNotebookでは置換ロジックを書き直さない。
後で `conv_tucker.py` に共通化したらimportへ置き換える。


## 3. 比較するrank候補を決める

対象はまず `conv2` 1層だけに固定する。

`rank_out` と `rank_in` を同時に変えると組合せが増えるため、
最初は少数の候補から始める。

各rankは元の `out_channels` / `in_channels` を超えないこと。


In [ ]:
rank_settings = [
    # TODO: 例を参考に、自分で比較したい(rank_out, rank_in)を数組決める。
    # {"rank_out": ..., "rank_in": ...},
]

rank_settings


## 4. 各rankで同じ実験を繰り返す

各候補について必ず同じ順番で比較する。

1. baselineをdeepcopy
2. `conv2` だけTucker-2へ置換
3. Tucker-2後のパラメータ数を数える
4. validation/test accuracyを測る
5. baselineからのaccuracy dropを求める
6. 結果を1行として保存する

最低限保存する列:

```text
rank_out
rank_in
parameters
parameters_reduction
validation_acc
accuracy_drop
```

必要ならweight再構成誤差も追加する。


In [ ]:
results = []

for setting in rank_settings:
    rank_out = setting["rank_out"]
    rank_in = setting["rank_in"]

    # TODO:
    # compressed_model = ...
    # validation_loss, validation_acc = ...
    # results.append({...})
    pass


## 5. 表にしてtrade-offを読む

DataFrameなどでrank候補を横並びにする。

「一番accuracyが高い」だけでも「一番小さい」だけでもなく、

```text
パラメータを大きく減らせる
かつ
accuracy dropが許容できる
```

候補を探す。


In [ ]:
# TODO:
# import pandas as pd
# df = pd.DataFrame(results)
# df


## 6. 可能ならPareto的に候補を見る

ある候補Aが候補Bより

- パラメータ数が少ない
- accuracyも高い

なら、Bを選ぶ理由は弱い。

最終的にfine-tuningへ持っていくrank候補を1〜数個選ぶ。
複雑な自動rank選択アルゴリズムはまだ作らなくてよい。


In [ ]:
# TODO:
# 結果をparameter数順やaccuracy順に並べて、候補を確認する。


## 7. このNotebookの完了条件

次を説明できれば `03_finetuning.ipynb` へ進む。

1. `rank_out` / `rank_in` を下げると何が小さくなるか
2. rankを下げすぎるとaccuracyが落ちる理由
3. パラメータ数だけでrankを選んではいけない理由
4. fine-tuningへ持っていくrank候補を根拠付きで選べる
